In [3]:
%pip install pandas matplotlib seaborn scikit-learn

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd

df = pd.read_csv('processed/transactions_train_filtered.csv')
df['t_dat'] = pd.to_datetime(df['t_dat'])
df = df.sort_values(by=['customer_id', 't_dat'])


FileNotFoundError: [Errno 2] No such file or directory: 'processed/transactions_train_filtered.csv'

In [ ]:
from collections import defaultdict

# For reproducibility, only keep customer_id and article_id
customer_sequences = defaultdict(list)

for row in df.itertuples(index=False):
    customer_sequences[row.customer_id].append(row.article_id)

sequences = list(customer_sequences.values())


In [ ]:
X, Y = [], []

for seq in sequences:
    if len(seq) < 14:
        continue
    for i in range(len(seq) - 14 + 1):
        X.append(seq[i:i+7])
        Y.append(seq[i+7:i+14])


In [ ]:
from sklearn.preprocessing import LabelEncoder

all_articles = pd.Series([item for sublist in (X + Y) for item in sublist])
le = LabelEncoder()
le.fit(all_articles)

X_encoded = [[le.transform([article])[0] for article in seq] for seq in X]
Y_encoded = [[le.transform([article])[0] for article in seq] for seq in Y]

num_articles = len(le.classes_)


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class ArticleDataset(Dataset):
    def __init__(self, X, Y):
        self.X = X
        self.Y = Y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.long), torch.tensor(self.Y[idx], dtype=torch.long)

dataset = ArticleDataset(X_encoded, Y_encoded)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)


In [ ]:
import torch.nn as nn

'''
class LSTMModel(nn.Module):
    def __init__(self, num_articles, embedding_dim=50, hidden_dim=100):
        super(LSTMModel, self).__init__()
        self.embedding = nn.Embedding(num_articles, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_articles)

    def forward(self, x):
        embedded = self.embedding(x)             # (batch, seq_len, embed_dim)
        lstm_out, _ = self.lstm(embedded)        # (batch, seq_len, hidden_dim)
        output = self.fc(lstm_out[:, -1, :])     # (batch, num_articles)
        return output
'''

#7/30 c
class LSTMModel(nn.Module):
    def __init__(self, num_articles, embedding_dim=50, hidden_dim=100, output_seq_len=7):
        super(LSTMModel, self).__init__()
        self.embedding = nn.Embedding(num_articles, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_articles * output_seq_len)
        self.output_seq_len = output_seq_len
        self.num_articles = num_articles

    def forward(self, x):
        embeddings = self.embedding(x)   
        lstm_out, _ = self.lstm(embeddings)  
        last_output = lstm_out[:, -1, :]   
        out = self.fc(last_output)   
        out = out.view(-1, self.output_seq_len, self.num_articles)  
        return out


In [ ]:
'''
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LSTMModel(num_articles).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x_batch, y_batch in dataloader:
        x_batch = x_batch.to(device)
        y_batch = y_batch[:, 0].to(device)  # Predict only the first of the 7 future articles

        optimizer.zero_grad()
        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss / len(dataloader)}")
'''
#7/30 c
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LSTMModel(num_articles, embedding_dim=50, hidden_dim=100, output_seq_len=7).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x_batch, y_batch in dataloader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)   

        optimizer.zero_grad()
        outputs = model(x_batch)   

        outputs_flat = outputs.view(-1, num_articles)   
        y_batch_flat = y_batch.view(-1)   

        loss = criterion(outputs_flat, y_batch_flat)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}")


In [ ]:
#Optional Saving the Model: torch.save(model.state_dict(), 'lstm_model.pth')


In [ ]:

'''
model.eval()
with torch.no_grad():
    test_input = torch.tensor(X_encoded[0], dtype=torch.long).unsqueeze(0).to(device)
    output = model(test_input)
    top_preds = torch.topk(output, 7).indices.squeeze(0).tolist()

    predicted_articles = le.inverse_transform(top_preds)
    print("Recommended next articles:", predicted_articles)
'''
#7/30 c
model.eval()
with torch.no_grad():
    test_input = torch.tensor(X_encoded[0], dtype=torch.long).unsqueeze(0).to(device)   
    output = model(test_input)   
    top_preds_per_step = torch.topk(output, k=1, dim=2).indices.squeeze(2)   
    predicted_articles = []
    for day_preds in top_preds_per_step[0]:
        predicted_articles.append(le.inverse_transform([day_preds.item()])[0])
    print("Recommended next 7 articles:", predicted_articles)
